# Tutorial 03: HLLLattice — Temporal Lattice of HLLSet Observations

This tutorial introduces the `HLLLattice` module, which implements a temporal lattice structure (the **W lattice**) for tracking HLLSet observations over time.

## What You'll Learn

1. **Lattice Concepts** — The W lattice and partial ordering
2. **LatticeNode** — Immutable, content-addressed nodes
3. **Creating Lattices** — Appending observations over time
4. **Lattice Operations** — Join (∨) and meet (∧)
5. **Temporal Queries** — Cumulative merge, delta, time ranges
6. **Storage Protocol** — Custom persistence backends

## Key Concepts

The W lattice is defined by the manuscript (§2.3):

> The set of all HLLSet fingerprints is partially ordered by bitwise inclusion:
> $A \leq B \iff R_A \land \neg R_B = 0$

This forms a **distributive lattice** with:
- **Meet (∧)**: Bitwise AND (intersection)
- **Join (∨)**: Bitwise OR (union)
- **Bottom**: Empty HLLSet (∅)
- **Top**: All-ones matrix

In [1]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.hll_lattice import HLLLattice, LatticeNode, InMemoryStorage
from core.hllset import HLLSet
import time

## 1. Lattice Concepts

Each time-step `t` produces a `LatticeNode`:

```
Lattice(t) = { T₁(t), T₂(t), ..., Tₖ(t) }   ← individual HLLSets
           ↓ merge (bitwise OR)
         M(t) = ∪ Tᵢ(t)                     ← merged snapshot
           ↓ content-address
       ID(t) = SHA1(M(t).registers)         ← node identity
```

Temporal ordering: `Lattice(t₁) ⊑ Lattice(t₂) ⟺ M(t₁) ⊆ M(t₂)`

In [2]:
# Create a lattice
lattice = HLLLattice(p_bits=10)

print(f"Lattice p_bits: {lattice.p_bits}")
print(f"Initial node count: {lattice.node_count}")

Lattice p_bits: 10
Initial node count: 0


## 2. LatticeNode

A `LatticeNode` is an immutable, content-addressed observation snapshot.

Each node contains:
- **node_id**: SHA1 hash of merged registers
- **timestamp**: When the node was created
- **merged**: The merged HLLSet
- **cardinality**: Estimated unique elements
- **popcount**: Total set bits (conserved quantity)
- **component_ids**: IDs of individual HLLSets that were merged
- **parent_ids**: Causal predecessor nodes

In [3]:
# Create HLLSets for observations
obs_1 = HLLSet.from_batch(["hello", "world"])
obs_2 = HLLSet.from_batch(["foo", "bar", "baz"])

# Append as a new node
node = lattice.append(
    hllsets=[obs_1, obs_2],
    timestamp=1.0,
    metadata={"source": "tutorial_example"}
)

print(f"Node ID: {node.node_id[:16]}...")
print(f"Timestamp: {node.timestamp}")
print(f"Cardinality: {node.cardinality:.0f}")
print(f"Popcount: {node.popcount}")
print(f"Component IDs: {len(node.component_ids)}")
print(f"Parent IDs: {node.parent_ids}")
print(f"Metadata: {node.metadata}")

Node ID: 3fcb85ff4a50dbb4...
Timestamp: 1.0
Cardinality: 5
Popcount: 5
Component IDs: 2
Parent IDs: ()
Metadata: {'source': 'tutorial_example'}


## 3. Creating Lattices

Append observations over time to build the temporal structure.

In [4]:
# Create a fresh lattice
lattice = HLLLattice(p_bits=10)

# Simulate observations over time
observations = [
    (1.0, [["event_a", "event_b"]]),
    (2.0, [["event_c", "event_d"], ["event_e"]]),
    (3.0, [["event_f", "event_a"]]),  # Note: "event_a" appears again
    (4.0, [["event_g", "event_h", "event_i"]]),
]

nodes = []
for ts, token_batches in observations:
    node = lattice.append_tokens(token_batches, timestamp=ts)
    nodes.append(node)
    print(f"t={ts}: node={node.node_id[:8]}..., |M|≈{node.cardinality:.0f}, pop={node.popcount}")

t=1.0: node=b9f4d460..., |M|≈3, pop=2
t=2.0: node=1a1265cd..., |M|≈4, pop=3
t=3.0: node=75865f9f..., |M|≈3, pop=2
t=4.0: node=e7941ead..., |M|≈4, pop=3


In [5]:
# Check lattice state
print(f"\nTotal nodes: {lattice.node_count}")
print(f"Latest node: {lattice.latest_node()}")


Total nodes: 4
Latest node: LatticeNode(e7941ead..., t=4.0, |M|≈4, pop=3)


## 4. Lattice Operations

The lattice supports two fundamental operations:

- **Join (∨)**: `a ∨ b = union` — combines observations
- **Meet (∧)**: `a ∧ b = intersection` — finds common elements

In [6]:
# Get two nodes
node_a = nodes[0]  # t=1.0
node_b = nodes[2]  # t=3.0 (shares "event_a")

print(f"Node A (t={node_a.timestamp}): |M|≈{node_a.cardinality:.0f}")
print(f"Node B (t={node_b.timestamp}): |M|≈{node_b.cardinality:.0f}")

Node A (t=1.0): |M|≈3
Node B (t=3.0): |M|≈3


In [7]:
# Lattice JOIN (∨) - union
joined = lattice.join(node_a, node_b)

print(f"Join (A ∨ B):")
print(f"  Node ID: {joined.node_id[:16]}...")
print(f"  Cardinality: {joined.cardinality:.0f}")
print(f"  Popcount: {joined.popcount}")
print(f"  Parents: {[p[:8] for p in joined.parent_ids]}")

Join (A ∨ B):
  Node ID: 202c071b3d31c664...
  Cardinality: 4
  Popcount: 3
  Parents: ['b9f4d460', '75865f9f']


In [8]:
# Lattice MEET (∧) - intersection
met = lattice.meet(node_a, node_b)

print(f"Meet (A ∧ B):")
print(f"  Node ID: {met.node_id[:16]}...")
print(f"  Cardinality: {met.cardinality:.0f}")
print(f"  Popcount: {met.popcount}")
print(f"  (Common element 'event_a' should be captured)")

Meet (A ∧ B):
  Node ID: 1aa2a650d9c57cb3...
  Cardinality: 2
  Popcount: 1
  (Common element 'event_a' should be captured)


In [9]:
# Subset checking
print(f"\nPartial Order:")
print(f"  A ⊆ Join(A,B): {node_a.is_subset_of(joined)}")
print(f"  B ⊆ Join(A,B): {node_b.is_subset_of(joined)}")
print(f"  Meet(A,B) ⊆ A: {met.is_subset_of(node_a)}")
print(f"  Meet(A,B) ⊆ B: {met.is_subset_of(node_b)}")


Partial Order:
  A ⊆ Join(A,B): True
  B ⊆ Join(A,B): True
  Meet(A,B) ⊆ A: True
  Meet(A,B) ⊆ B: True


## 5. Temporal Queries

The lattice supports temporal aggregation and queries:

- **cumulative(t)**: Union of all observations up to time `t`
- **delta(t1, t2)**: What changed between `t1` and `t2`
- **nodes_in_range(t1, t2)**: All nodes in time range

In [10]:
# Cumulative merge up to time t
cum_t2 = lattice.cumulative(t=2.0)
cum_t4 = lattice.cumulative(t=4.0)
cum_all = lattice.cumulative()  # All time

print(f"Cumulative cardinality:")
print(f"  Up to t=2.0: ~{cum_t2.cardinality():.0f}")
print(f"  Up to t=4.0: ~{cum_t4.cardinality():.0f}")
print(f"  All time:    ~{cum_all.cardinality():.0f}")

Cumulative cardinality:
  Up to t=2.0: ~5
  Up to t=4.0: ~10
  All time:    ~10


In [11]:
# Delta: what changed between time points?
delta = lattice.delta(t1=2.0, t2=4.0)

print(f"Delta (t=2.0 → t=4.0):")
print(f"  New information: ~{delta.cardinality():.0f} elements")

Delta (t=2.0 → t=4.0):
  New information: ~5 elements


In [12]:
# Query nodes in time range
range_nodes = lattice.nodes_in_range(t_start=1.5, t_end=3.5)

print(f"Nodes in range [1.5, 3.5]:")
for n in range_nodes:
    print(f"  t={n.timestamp}: {n.node_id[:12]}...")

Nodes in range [1.5, 3.5]:
  t=2.0: 1a1265cd2cef...
  t=3.0: 75865f9f75ef...


In [13]:
# Get all nodes
all_nodes = lattice.all_nodes()
print(f"\nAll {len(all_nodes)} nodes (ordered by time):")
for n in all_nodes:
    print(f"  t={n.timestamp}: |M|≈{n.cardinality:.0f}")


All 4 nodes (ordered by time):
  t=1.0: |M|≈3
  t=2.0: |M|≈4
  t=3.0: |M|≈3
  t=4.0: |M|≈4


## 6. Node Lookup

Nodes can be retrieved by their content-addressed ID.

In [14]:
# Lookup by ID
target_id = nodes[1].node_id
retrieved = lattice.node_by_id(target_id)

if retrieved:
    print(f"Found node: {retrieved}")
    print(f"  Timestamp: {retrieved.timestamp}")
    print(f"  Components: {len(retrieved.component_ids)}")
else:
    print("Node not found")

Found node: LatticeNode(1a1265cd..., t=2.0, |M|≈4, pop=3)
  Timestamp: 2.0
  Components: 2


In [15]:
# Non-existent ID
missing = lattice.node_by_id("0" * 40)
print(f"Missing node: {missing}")

Missing node: None


## 7. Causality and Parent Relationships

Each node tracks its causal predecessors via `parent_ids`.

In [16]:
# View parent relationships
print("Parent relationships (chronological):")
for i, n in enumerate(nodes):
    parent_str = "[root]" if not n.parent_ids else f"[{n.parent_ids[0][:8]}...]"
    print(f"  Node {i} (t={n.timestamp}): parents={parent_str}")

Parent relationships (chronological):
  Node 0 (t=1.0): parents=[root]
  Node 1 (t=2.0): parents=[b9f4d460...]
  Node 2 (t=3.0): parents=[1a1265cd...]
  Node 3 (t=4.0): parents=[75865f9f...]


In [ ]:
# Custom parent specification
# Create a node that explicitly branches from node 1
branch_node = lattice.append(
    hllsets=[HLLSet.from_batch(["branch_event"])],
    timestamp=5.0,
    parent_ids=[nodes[1].node_id],  # Branch from t=2.0
    metadata={"branch": True}
)

print(f"Branch node parents: {[p[:8] for p in branch_node.parent_ids]}")
print(f"(Branches from node at t=2.0)")

## 8. Storage Protocol

The lattice uses a pluggable storage protocol. The default is `InMemoryStorage`, but you can implement custom backends.

In [17]:
# View storage interface
storage = InMemoryStorage()

# Store a node
test_hll = HLLSet.from_batch(["test"])
test_lattice = HLLLattice(p_bits=10, storage=storage)
test_node = test_lattice.append([test_hll])

print(f"Storage node count: {storage.node_count()}")
print(f"Stored node IDs: {storage.list_node_ids()[:3]}...")

Storage node count: 1
Stored node IDs: ['1131f8d907f1f1a349b6cb4c142bbd56127dadf6']...


In [18]:
# Load from storage
loaded = storage.load_node(test_node.node_id)
print(f"Loaded node: {loaded}")
print(f"Same as original: {loaded == test_node}")

Loaded node: LatticeNode(1131f8d9..., t=1775928767.5, |M|≈2, pop=1)
Same as original: True


## 9. Practical Example: Event Stream Processing

Let's simulate processing an event stream with the lattice.

In [19]:
import random

# Create lattice for event processing
event_lattice = HLLLattice(p_bits=10)

# Simulate hourly event batches
event_types = ["login", "logout", "purchase", "view", "click", "search"]

for hour in range(1, 25):  # 24 hours
    # Generate random events for this hour
    num_events = random.randint(50, 200)
    events = [
        f"user_{random.randint(1, 100)}_{random.choice(event_types)}"
        for _ in range(num_events)
    ]
    
    # Append to lattice
    node = event_lattice.append_tokens(
        [events], 
        timestamp=float(hour),
        metadata={"hour": hour, "raw_count": num_events}
    )

print(f"Processed 24 hours of events")
print(f"Total nodes: {event_lattice.node_count}")

Processed 24 hours of events
Total nodes: 24


In [20]:
# Analyze patterns
print("\nHourly unique events:")
for node in event_lattice.all_nodes()[:6]:  # First 6 hours
    hour = node.metadata.get("hour", "?")
    raw = node.metadata.get("raw_count", "?")
    print(f"  Hour {hour}: ~{node.cardinality:.0f} unique (from {raw} raw)")


Hourly unique events:
  Hour 1: ~116 unique (from 121 raw)
  Hour 2: ~171 unique (from 195 raw)
  Hour 3: ~131 unique (from 144 raw)
  Hour 4: ~144 unique (from 159 raw)
  Hour 5: ~129 unique (from 139 raw)
  Hour 6: ~68 unique (from 77 raw)


In [21]:
# Cumulative unique events over time
checkpoints = [6, 12, 18, 24]
print("\nCumulative unique events:")
for t in checkpoints:
    cum = event_lattice.cumulative(t=float(t))
    print(f"  After {t} hours: ~{cum.cardinality():.0f} unique events")


Cumulative unique events:
  After 6 hours: ~472 unique events
  After 12 hours: ~593 unique events
  After 18 hours: ~603 unique events
  After 24 hours: ~608 unique events


In [22]:
# What's new in each time period?
print("\nNew events per period:")
periods = [(0, 6), (6, 12), (12, 18), (18, 24)]
for t1, t2 in periods:
    delta = event_lattice.delta(float(t1), float(t2))
    print(f"  Hours {t1}-{t2}: ~{delta.cardinality():.0f} new events")


New events per period:
  Hours 0-6: ~472 new events
  Hours 6-12: ~110 new events
  Hours 12-18: ~15 new events
  Hours 18-24: ~5 new events


## Summary

In this tutorial, you learned:

1. **W Lattice** — Partial order on HLLSet fingerprints
2. **LatticeNode** — Immutable, content-addressed snapshots
3. **Appending** — Building temporal structure from observations
4. **Join/Meet** — Lattice operations (∨ = union, ∧ = intersection)
5. **Temporal Queries** — Cumulative, delta, time ranges
6. **Storage Protocol** — Pluggable persistence backends

### Key Theorem (Corollary 2.4)

> In a closed environment where additions and deletions balance,
> the W lattice is **INVARIANT** — its structure is conserved.

### Next Steps

- **Tutorial 04**: HLLSetStore — Persistent storage with derivation tracking